In [2]:
%cd ../.
import os, sys
sys.path.insert(0, os.path.expanduser('~/CDD_Vault_API/python'))  # CDD Vault API (get_df)

/home/gtamo/Px_interface


In [3]:
%load_ext autoreload
%autoreload 2

import os, re, gc, ctypes, json, time, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import date
from rdkit import Chem
import yaml
import joblib
from tqdm import tqdm
from types import SimpleNamespace
tqdm.pandas()

# local modules
import python.functions as fn
import python.Px_interface as px
from get_library import get_df   # CDD Vault collection export

# pandas display options:
pd.set_option('display.max_colwidth', None)

In [4]:
## reuse the CLI classes and the publish functions from python/Px_interface.py (%autoreload picks up edits)
from python.Px_interface import PARAMS, DATA, OUTPUT, aws_publish_targets, publish_interface

CONFIG = 'config/config.yaml'   # swap to 'tmp/config.yaml' to debug on the synthetic fixture
OUTPUT_DIR = 'output'          # local base dir for the HTML + volcanoes (interfaces/ created under it), as the CLI default

## params (load_params also binds the Dropbox readers params.dbx / dbx_glob / dbx_push: rclone on the cluster):
params = PARAMS(CONFIG)
params.load_params()

## show:
print('> chemlib overwrite:', params.CHEMLIB_OVERWRITE)
print('> dfraw overwrite:', params.DFRAW_OVERWRITE)
print('> iface overwrite:',params.IFACE_OVERWRITE)
print('> srb_png update:',params.UPDATE_PNGS)

print('> njobs', params.NJOBS)

> Dropbox backend: rclone
> chemlib overwrite: False
> dfraw overwrite: False
> iface overwrite: False
> srb_png update: False
> njobs 16


In [5]:
## data:
data = DATA()
data.load_chemical_lib_df(params)
data.download_cdd_pngs(params)   # refresh SRB_PNG_DIR from CDD when UPDATE_PNGS=true (else no-op)
data.load_old_df(params)
data.load_new_df(params)
data.get_contaminants_and_controls(params)
data.get_gene_research(params)

> Chemical lib dim: (14168, 13)
> df_raw (24349157, 13) | MS (2277, 5)
> RDS measure: 5,097 experiments from the database + 1,570 from the archive | 1,194 database experiments without a date skipped
> RDS report: 5,097 experiments from the database + 1,570 from the archive | 1,194 database experiments without a date skipped
> association: RDS opentarget, 30,468 genes (+ px_ms_score_glues for the validation targets)
> FBX: MEASURE 56,698,074 rows | MSSCORE 343,672 | REPORT 6,667 (6,667 experiments) | validation MS score: 40 targets
> loaded degradation research for 10191 genes


In [6]:
## output:
output = OUTPUT()
output.combine_datasets(data, params)
output.get_de_validated(data, params)
output.get_iface(data, params)

> combined MEASURE: 80,230,546 rows | 9,074 experiments | 7,305 compounds | 13,264 genes
> combined MS-SCORE: 391,539 (gene,compound) rows | 236,061 (gene,plate) | FBX 341,902, df_raw 49,637
> combined REPORT: 9,074 uniquecontrasts | compounds 7,305 | plates 311
> plate dates: 311 plates mapped | report spans 2026-04-29 .. 2026-10-06
> categorized string columns | MEASURE now 13.8 GB
> freed FBX_* + MS sources (FREE_UPSTREAM)
> target: 40 validated - 2772 devalidated targets
> compound: 93 validated - 436 devalidated compounds
> loaded interface inputs from output/interface/ | iface_df (11342, 6), compounds_df (388706, 13), meas (80230546, 7), 311 plate dates


In [7]:
## Build interface separately to avoid re-running all script
output.build_interface(data, params, OUTPUT_DIR)

> Plates default-ticked: 38 plate(s) in validation 2026-10-06
> gene_research: 10191 genes
> 11,342 / 11,342 genes after filtering (x=R2, y=association_score, z=ms_score)
  [highlight] corner-top-40=40, association_score>None: 0, must=33, union=71
  [range_sliders] panels built for all 11342 plotted genes
> panels: loaded 11,342 gene panels from cache (skipped rebuild)
> no compound panel (provide compounds_df or top1..topN columns) — scatter + hover text only
> stem trace: 11,751 gene positions across 522 validation contrasts
> volcano gene search: 13,258 genes x 7,171 volcanoes (7,171 file(s) written, 0 cached) -> output/interfaces/volcanoes_px/gpos
  [range_sliders] default box ≈ 11342 genes; gene labels shown while ≤ 1000 in range (drag handles to widen/narrow)
  [activity_defaults] focus view starts on ['Low (2-10)', 'Single (1)']
  [gene_research] 9208/10191 records kept (plotted genes only) — 23.6 MB injected
wrote output/interfaces/Serac_Px_interface.html  (3.6 MB main doc)  + 

In [8]:
## Publish the build to AWS (S3, then the EC2)
publish_interface(OUTPUT_DIR, params, aws_publish_targets(params))

> publish target: s3://vpn-project-interface-620423424620/interfaces/ -> EC2 i-08fa9cd5f078fd225 (SSM Online)
> publish: 20223 files (749 MB) in S3 after 54s
> publish: EC2 synced after 61s (files=20223) -> https://advantedge.seracbio.com/Px_interface/


In [9]:
## upload the combined tables to Dropbox (rclone on the cluster); config EXPORT_COMBINED does this during the build
# params.dbx_push(output.measure, params.DROPBOX_DANIELA + 'Px_parquet_from_GT/Px_MEASURE.parquet')
# params.dbx_push(output.mscore, params.DROPBOX_DANIELA + 'Px_parquet_from_GT/Px_MSCORE.parquet')
# params.dbx_push(output.report, params.DROPBOX_DANIELA + 'Px_parquet_from_GT/Px_REPORT.parquet')